# FDM modeling of ARK-cross sections

We use the developed code from "src/ARK_geotop.py"

In [50]:
import os
import sys
from typing import Any
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pdf2image
import pickle

from tools.fdm.src.mfgrid import Grid
from tools.fdm.src.fdm3 import fdm3

from mf6lab.Projects.ARK_RWS.src.ARK_geotop import (
    parse_geotop_filename,
    Geotop_xsec,
    CrossSectionDigitizer,
    ImagePicker,
    Dirs,
    plot_result,    
    LITHO_CLASSES,
    GEO_UNITS
    )

print(sys.executable)

# --- Needed to make figure separate from the notebook and interactive
%matplotlib qt

# --- Seet the namespace for the relevant directories
dirs = Dirs()

# --- Get the paths and names of  the geotop pdf files in the order they are in dirs.dino
xsec_paths = {i:name for i, name in enumerate(glob(dirs.dino + '*.pdf'))}
xsec_names = {i:os.path.basename(name) for i, name in enumerate(glob(dirs.dino + '*.pdf'))}

# --- Pickling
def pickleto(var:Any, basename:str, parent:str=dirs.data):
    """Pickle var to os.path.join(dirs.data, basename)"""
    if not basename.endswith('.pkl'):
        basename += ".pkl"

    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'wb') as f:
        print(f"Pickled {basename} --> {parent}")        
        pickle.dump(var, f)

# --- Unpickling
def picklefrom(basename:str, parent:str=dirs.data)->Any:
    """Unpickle varname from os.path.join(parent, basename)"""
    if not basename.endswith(".pkl"):
        basename += ".pkl"
          
    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'rb') as f:
        print(f"Loaded {basename} <-- {parent}")        
        return pickle.load(f)


def spy(idx_arr):
    """Show where the index labels are in the xsec idx_arr."""
    fig, ax = plt.subplots(figsize=(10, 6))    
    ax.set_title("Location of legend indices in xsec.idx_arr")
    classes = np.unique(idx_arr)
    cmap = plt.get_cmap('tab20', len(classes) - 1)
    mappable = ax.imshow(idx_arr, cmap=cmap, origin='upper')
    fig.colorbar(mappable)
    plt.show()

# --- Color for empty legend (empty voxel with geo_unit 'none')
WHITE_01 = np.array([1., 1., 1.])

/Users/Theo/Development/python/mf6_tools/mf6lab/.venv/bin/python


In [5]:
xsec_names

{0: 'BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf',
 1: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf',
 2: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130049,479466.pdf',
 3: 'BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf',
 4: 'BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf',
 5: 'BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf',
 6: 'BRO GeoTOP Verticale doorsnede geologische eenheid 126311,474187.pdf',
 7: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 129926,479462.pdf',
 8: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 127484,477893.pdf',
 9: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130173,479431.pdf'}

In [6]:
geotop_xsecs = picklefrom("geotop_xsecs.pkl")

Loaded geotop_xsecs.pkl <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/


## Deal with the second cross section only

In [84]:
isec = 1
xsec = geotop_xsecs[xsec_names[isec]]

print(f"Dealing with xsec {isec}:\n{xsec.name}") 

# --- find the ARK it wronly has index 1 in row 6
ix_ARK = np.where(xsec.idx_arr[6] == 1)[0]

spy(xsec.idx_arr)


Dealing with xsec 1:
BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf


In [85]:
idx_arr = xsec.idx_arr.copy()

idx_arr[:3].T


array([[0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 3],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 3],
       [0, 1, 3],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 1, 1],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 1, 1],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 3],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 6, 3],
       [0, 3, 3],
       [0, 0, 3],
       [0, 0, 1],
       [0, 0, 0],
       [0, 0, 0],
       [0,

### Repair the idx_arr for the ARK canal which looks now filled with material 'a'

The problem is that the idx_arr contains index 1 (anthropogenic) inside the ARK,
which was likely caused by some vertical grid line disturbing the color_match,
so the light gray legend color (1) was matched instead of white (0).

To find ARK look for index = 1 in row 6 to find ARK incision in the xsec.
Then replace all the 1 indices in that column with 0 to make it empty.

In [ ]:
idx_arr = xsec.idx_arr.copy()

# --- The columns have idx 1 incorrectly
cols = np.where(idx_arr[1] == 1)[0]

# --- Replace by index 0 ('none')
for j in cols:
    rows = idx_arr[:, j] == 1
    idx_arr[rows, j] = 0
    
# --- Check
spy(idx_arr)

When this works replace the xsec.idx_arr with the corrected version

In [ ]:
# --- Replace origional idx_arr by corrected one
xsec.idx_arr = idx_arr

# --- Check
spy(xsec.idx_arr)
print('idx_arr repaired')

## Model grid

In [124]:
def Ifrom_v_extent(gr, extent):
    """Return global index given vertical extent = (xmin, xmax, zmin, zmax) """
    xmin, xmax, zmin, zmax = extent
    mask = np.logical_and.reduce([
        gr.XM > xmin, gr.XM < xmax,
        gr.ZM > zmin, gr.ZM < zmax
    ])
    return gr.NOD[mask]


In [ ]:
# --- Cross section data
GROUND_ELEV = -1.3
ARK_STAGE = -0.4
D_DAMW = 0.5
Z_DAMW = -12.

# --- index arrays
ixARK = np.where(xsec.idx_arr[10] == 0)[0]
izARK = np.where(xsec.idx_arr[:, ixARK] == 0)[0]

# --- ARK xsec extent
ARK_extent=np.array([xsec.x[ixARK[0]], xsec.x[ixARK[-1] + 1],
                     xsec.z[izARK.max()], ARK_STAGE])
xARKmin, xARKmax, zARKmin, zARKmax = ARK_extent
damw_west_extent = np.array([xARKmin - D_DAMW, xARKmin, Z_DAMW, 0])
damw_east_extent = np.array([xARKmax, xARKmax + D_DAMW, Z_DAMW, 0])

# --- Grid
x = xsec.x, detail ARK fi
gr = Grid(x, None, xsec.z)

IBOUND = gr.const(1, dtype=int)
IBOUND[gr.ZM > GROUND_ELEV] = 0

# --- Get the global cell indices
Idw = Ifrom_v_extent(gr, damw_west_extent)
Ide = Ifrom_v_extent(gr, damw_east_extent)
Iark = Ifrom_v_extent(gr, ARK_extent)
    

# --- Set to fixed head
IBOUND.ravel()[Iark] = -1

# --- Set polder water level
POLDERPEIL = GROUND_ELEV - 1.0

# --- Cells with POLDERPEIL
mask_ARK = gr.const(0, dtype=bool)
mask_ARK.ravel()[Iark] = True
Ipp = gr.NOD[np.logical_and(np.abs(gr.ZM - POLDERPEIL) <= xsec.dz / 2, ~mask_ARK)]


In [127]:
Ipp

array([228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240,
       241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253,
       254, 255, 256, 257, 258, 260, 261, 262, 263, 264, 265, 266, 267,
       268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280,
       281, 282, 283, 284])

In [ ]:
gr.idx_arr= xsec.overlap(gr)
props = xsec.get_props(gr.idx_arr)

# Patches
water = np.array(xwmin, xwmax, zwmin, zwmax)
sheetL = np.array(xsh...)
sheetR = np.array()

# Polypatch
tzone = polyline


kh = gr.patch(kh, water)
kh = gr.patch(kh, sheet)

K = (props['kh'], props['kh'], props['kv'])
DRN = {}
CHD = {}

out = fdm3(K=...)

